# Gold - Fato Entregas

Monitoramento e métricas de desempenho de entregas (delivery KPIs) e cálculo de atrasos.

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
table_name = 'fact_entregas'
output_path_data = f"{var_gold}/{table_name}/data"
table_name_schema = f'{var_environment}.{var_gold_schema}.{table_name}'

In [ ]:
from pyspark.sql.functions import col, sha2, date_format, coalesce, lit, when, datediff

df_entregas = spark.read.table(f"{var_environment}.{var_silver_schema}.case_logistica_entregas")
df_cab = spark.read.table(f"{var_environment}.{var_silver_schema}.case_erp_pedidos_cabecalho")
df_cli = spark.read.table(f"{var_environment}.{var_gold_schema}.dim_clientes")

df_fact = (
    df_entregas
    .join(df_cab, "id_pedido", "left")
    .join(df_cli, "id_cliente", "left")
    .withColumn("sk_cliente", coalesce(col("sk_cliente"), sha2(lit("-1"), 256)))
    .withColumn("sk_tempo_envio", coalesce(date_format(col("data_envio"), "yyyyMMdd").cast("integer"), lit(-1)))
    .withColumn("sk_tempo_entrega", coalesce(date_format(col("data_entrega"), "yyyyMMdd").cast("integer"), lit(-1)))
    
    # Cálculo logístico
    .withColumn("dias_transporte", datediff(col("data_entrega"), col("data_envio")))
    .withColumn("flag_atrasado", when(col("status_entrega") == "Atrasado", lit(1)).otherwise(lit(0)))
    
    .select(
        col("id_entrega").alias("id_fato_entrega"),
        col("id_pedido"),
        col("sk_cliente"),
        col("sk_tempo_envio"),
        col("sk_tempo_entrega"),
        col("status_entrega"),
        col("transportadora"),
        col("modalidade_transporte"),
        col("custo_frete"),
        col("dias_transporte"),
        col("flag_atrasado")
    )
)

In [ ]:
df_write_fact = df_fact.withColumn("data_key_str", when(col("sk_tempo_entrega") != -1, col("sk_tempo_entrega").cast("string")).otherwise(col("sk_tempo_envio").cast("string")))

import pyspark.sql.functions as F

process_fact(
    df_write=df_write_fact,
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    data_formatada="data_key_str",
    chave_clusterby=["sk_tempo_envio"]
)